In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\5. Others\\1_segnet_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\5. Others\1_segnet_pytorch


In [2]:
# Demo (PyTorch)
# based on https://github.com/imlab-uiip/keras-segnet

# Overview
- SegNet for image segmentation (PyTorch)
- Input: depth image (acquired via Kinect camera), shape `(1, 96, 96)`
- Output: 7-class pixel-wise segmentation of hand components (palm, 5 fingers, background)

# Import libraries

In [3]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [4]:
n_classes = 7
kernel = 3
img_h = img_w = 96

# (1) Data

#### Hand images: https://drive.google.com/file/d/1MRiBnDW9dE7mNb5bL-jzhmtTd2CSklvL/view?usp=sharing

In [5]:
CLASS_COLOR_MAP = {
    255: 0,  # palm
    43:  1,  # finger 1
    85:  2,  # finger 2
    128: 3,  # finger 3
    170: 4,  # finger 4
    212: 5,  # finger 5
    0:   6,  # background
}

def gray2class_index(label_img, img_h=96, img_w=96):
    # Trả về mảng chỉ số lớp (H, W) kiểu int64 cho nn.CrossEntropyLoss
    class_map = np.full((img_h, img_w), fill_value=6, dtype=np.int64)
    for color, cls_idx in CLASS_COLOR_MAP.items():
        class_map[label_img == color] = cls_idx
    return class_map


def prepare_segnet_data(root_path):
    loi = sorted(os.listdir(root_path))

    # PyTorch NCHW: (N, 1, H, W) và nhãn pixel (N, H, W)
    data_x = np.zeros((len(loi), 1, img_h, img_w), dtype=np.float32)
    data_y = np.zeros((len(loi), img_h, img_w), dtype=np.int64)

    for im_ind, fn in enumerate(loi):
        full_fn = os.path.join(root_path, fn)
        labels_fn = full_fn.replace('depth', 'labels')

        img = np.uint8(resize(imread(full_fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))
        lbl_raw = np.uint8(resize(imread(labels_fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))

        data_x[im_ind, 0, :, :] = img
        data_y[im_ind, :, :] = gray2class_index(lbl_raw, img_h=img_h, img_w=img_w)

    return data_x, data_y

In [6]:
root_path = './data/segnet/train/depth'
x_train, y_train = prepare_segnet_data(root_path)

root_path = './data/segnet/test/depth'
x_test, y_test = prepare_segnet_data(root_path)

print('x_train:', x_train.shape, '| y_train:', y_train.shape)
print('x_test :', x_test.shape,  '| y_test :', y_test.shape)

x_train: (2450, 1, 96, 96) | y_train: (2450, 96, 96)
x_test : (1050, 1, 96, 96) | y_test : (1050, 96, 96)


In [7]:
sample_idx = min(1000, x_test.shape[0] - 1)
plt.imshow(x_test[sample_idx, 0, :, :], cmap='gray')
plt.title('Depth input')
plt.show()

plt.figure(figsize=(18, 3))
for cls_idx in range(n_classes):
    plt.subplot(1, n_classes, cls_idx + 1)
    plt.imshow((y_test[sample_idx] == cls_idx).astype(np.uint8), cmap='gray')
    plt.title(f'Class {cls_idx}')
    plt.axis('off')
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15452\1754570065.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


# (2) Declare model

In [8]:
def conv_bn_relu(in_ch, out_ch, k=3):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=k // 2),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class SegNet(nn.Module):
    def __init__(self, in_channels=1, n_classes=7, kernel=3):
        super().__init__()
        # Encoder (5 blocks VGG-16 style: 96 -> 48 -> 24 -> 12 -> 6 -> 3)
        self.encoder = nn.Sequential(
            conv_bn_relu(in_channels, 64, kernel),
            conv_bn_relu(64, 64, kernel),
            nn.MaxPool2d(2, 2),

            conv_bn_relu(64, 128, kernel),
            conv_bn_relu(128, 128, kernel),
            nn.MaxPool2d(2, 2),

            conv_bn_relu(128, 256, kernel),
            conv_bn_relu(256, 256, kernel),
            conv_bn_relu(256, 256, kernel),
            nn.MaxPool2d(2, 2),

            conv_bn_relu(256, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            nn.MaxPool2d(2, 2),

            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            nn.MaxPool2d(2, 2),
        )

        # Decoder (5 blocks: 3 -> 6 -> 12 -> 24 -> 48 -> 96)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),

            nn.Upsample(scale_factor=2, mode='nearest'),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 512, kernel),
            conv_bn_relu(512, 256, kernel),

            nn.Upsample(scale_factor=2, mode='nearest'),
            conv_bn_relu(256, 256, kernel),
            conv_bn_relu(256, 256, kernel),
            conv_bn_relu(256, 128, kernel),

            nn.Upsample(scale_factor=2, mode='nearest'),
            conv_bn_relu(128, 128, kernel),
            conv_bn_relu(128, 64, kernel),

            nn.Upsample(scale_factor=2, mode='nearest'),
            conv_bn_relu(64, 64, kernel),
            nn.Conv2d(64, n_classes, kernel_size=1, padding=0),
            nn.BatchNorm2d(n_classes),
        )

    def forward(self, x):
        x = self.encoder(x)
        logits = self.decoder(x)  # (B, n_classes, H, W)
        return logits

autoencoder = SegNet(in_channels=1, n_classes=n_classes, kernel=kernel).to(device)
trainable_params = sum(p.numel() for p in autoencoder.parameters() if p.requires_grad)
bn_buffers = sum(
    b.numel() for name, b in autoencoder.named_buffers()
    if name.endswith('running_mean') or name.endswith('running_var')
)
print(f'Trainable params     : {trainable_params:,}')      # Khớp chính xác 29,442,325 của bản Keras
print(f'BN running stats     : {bn_buffers:,}')            # Khớp chính xác 15,886 của bản Keras
print(f'Total (Keras-equiv)  : {trainable_params + bn_buffers:,}')  # Khớp chính xác 29,458,211

Trainable params     : 29,442,325
BN running stats     : 15,886
Total (Keras-equiv)  : 29,458,211


In [9]:
try:
    from torchinfo import summary
    print(summary(autoencoder, input_size=(1, 1, img_h, img_w), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
SegNet                                   [1, 7, 96, 96]            --
├─Sequential: 1-1                        [1, 512, 3, 3]            --
│    └─Sequential: 2-1                   [1, 64, 96, 96]           --
│    │    └─Conv2d: 3-1                  [1, 64, 96, 96]           640
│    │    └─BatchNorm2d: 3-2             [1, 64, 96, 96]           128
│    │    └─ReLU: 3-3                    [1, 64, 96, 96]           --
│    └─Sequential: 2-2                   [1, 64, 96, 96]           --
│    │    └─Conv2d: 3-4                  [1, 64, 96, 96]           36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 96, 96]           128
│    │    └─ReLU: 3-6                    [1, 64, 96, 96]           --
│    └─MaxPool2d: 2-3                    [1, 64, 48, 48]           --
│    └─Sequential: 2-4                   [1, 128, 48, 48]          --
│    │    └─Conv2d: 3-7                  [1, 128, 48, 48]          73,856
│   

# (3) Train defined model
- Nếu gặp lỗi tràn VRAM (OOM) trên GPU 4GB, giảm `batch_size` xuống `4`

In [10]:
def evaluate_segnet(model, loader, criterion, device):
    model.eval()
    total_loss, correct_pixels, total_pixels, total_imgs = 0.0, 0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            preds = logits.argmax(dim=1)
            correct_pixels += (preds == by).sum().item()
            total_pixels += by.numel()
            total_imgs += bx.size(0)
    return total_loss / total_imgs, correct_pixels / total_pixels


lr = 0.0001
batch_size = 8
nb_epoch = 15

criterion = nn.CrossEntropyLoss()  # Hỗ trợ trực tiếp (B, C, H, W) vs (B, H, W)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=lr)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

start_time = time.time()
for epoch in range(1, nb_epoch + 1):
    autoencoder.train()
    running_loss, correct_pixels, total_pixels, total_imgs = 0.0, 0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = autoencoder(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct_pixels += (logits.argmax(dim=1) == by).sum().item()
        total_pixels += by.numel()
        total_imgs += bx.size(0)

    train_loss = running_loss / total_imgs
    train_acc = correct_pixels / total_pixels
    val_loss, val_acc = evaluate_segnet(autoencoder, test_loader, criterion, device)
    print(f'Epoch {epoch:2d}/{nb_epoch} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

training_time = time.time() - start_time
print(f'> training time: {training_time:.2f}s')

Epoch  1/15 - loss: 1.1713 - acc: 0.6826 - val_loss: 1.0052 - val_acc: 0.7359


Epoch  2/15 - loss: 0.8962 - acc: 0.8024 - val_loss: 0.9626 - val_acc: 0.7652


Epoch  3/15 - loss: 0.7734 - acc: 0.8464 - val_loss: 0.9098 - val_acc: 0.7820


Epoch  4/15 - loss: 0.6938 - acc: 0.8705 - val_loss: 0.8533 - val_acc: 0.7944


Epoch  5/15 - loss: 0.6344 - acc: 0.8856 - val_loss: 0.8085 - val_acc: 0.8050


Epoch  6/15 - loss: 0.5866 - acc: 0.8961 - val_loss: 0.7773 - val_acc: 0.8107


... (5 outputs left out)

Epoch 12/15 - loss: 0.4139 - acc: 0.9217 - val_loss: 0.6918 - val_acc: 0.8169


Epoch 13/15 - loss: 0.3937 - acc: 0.9238 - val_loss: 0.6904 - val_acc: 0.8140


Epoch 14/15 - loss: 0.3719 - acc: 0.9273 - val_loss: 0.6847 - val_acc: 0.8159


Epoch 15/15 - loss: 0.3549 - acc: 0.9291 - val_loss: 0.6650 - val_acc: 0.8239
> training time: 415.62s


# (4) Evaluate trained model

In [11]:
test_loss, test_acc = evaluate_segnet(autoencoder, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

Test loss: 0.6649922850018456
Test accuracy: 0.8238987061838624


In [12]:
def class_to_rgb(output_dis):
    # Sửa np.float (deprecated trong NumPy >= 1.24) thành np.float32
    rgb_out = np.ones((img_h, img_w, 3), dtype=np.float32)

    rgb_out[output_dis == 0] = [0.0, 0.0, 0.0]
    rgb_out[output_dis == 1] = [0.0, 0.0, 0.5]
    rgb_out[output_dis == 2] = [0.0, 0.5, 0.0]
    rgb_out[output_dis == 3] = [0.5, 0.0, 0.0]
    rgb_out[output_dis == 4] = [0.5, 0.0, 0.5]
    rgb_out[output_dis == 5] = [0.5, 0.5, 0.0]
    rgb_out[output_dis == 6] = [1.0, 1.0, 1.0]

    return rgb_out

In [13]:
ran_ind = np.random.choice(x_test.shape[0], 1)[0]
sample = x_test[ran_ind]  # (1, 96, 96)

autoencoder.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(sample, axis=0)).to(device)
    pred_classes = autoencoder(inp).argmax(dim=1)[0].cpu().numpy()  # (96, 96)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample[0], cmap='gray')
plt.title('Input depth')

plt.subplot(1, 2, 2)
plt.imshow(class_to_rgb(pred_classes))
plt.title('SegNet prediction')
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15452\1754570065.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
